In [ ]:
!pip install torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -r requirements.txt

In [ ]:
import joblib
import os
import sys
import torch
import time
import boto3
import numpy as np
sys.modules.pop("models", None)
sys.modules.pop("prediction", None)
from functionality.prediction import Predictor
import pandas as pd
from xgboost import XGBClassifier
os.environ["TOKENIZERS_PARALLELISM"] = "true"
from functionality.models import ChemBertBinaryClassifier, MolFormerClassifier

In [103]:
protein_name = 'HSA'

In [36]:
#Load models
chembert = ChemBertBinaryClassifier.load_from_checkpoint(f"./models/{protein_name}_ChemBert.ckpt")
molformer = MolFormerClassifier.load_from_checkpoint(f"./models/{protein_name}_MolFormer.ckpt")
        
local_lightgbm_path = f"/models/{protein_name}_lightgbm.pkl"
lightgbm_model = joblib.load(local_lightgbm_path)

Model ./models/BRD4_CNN.ckpt loaded successfully!
Model ./models/BRD4_ChemBert.ckpt loaded successfully!
Model ./models/BRD4_MolFormer.ckpt loaded successfully!


In [ ]:
train_data_path = f'./train_data/{protein_name}/{protein_name}_train.parquet'
train_data = pd.read_parquet(train_data_path)[['molecule_smiles', 'binds']]
val_data_path = f'./train_data/{protein_name}/{protein_name}_val.parquet'
val_data = pd.read_parquet(val_data_path)[['molecule_smiles', 'binds']]
predictor = Predictor()

In [ ]:
chembert_proba = predictor.emb_prediction('ChemBert', train_data, chembert)
molformer_proba = predictor.emb_prediction('MolFormer', train_data, molformer)
lightgbm_proba = predictor.fp_prediction(train_data['molecule_smiles'], train_data['id'], lightgbm_model)

meta_features = np.column_stack(
                (
                    chembert_proba,
                    molformer_proba,
                    lightgbm_proba
                )
            )

In [109]:
start_time = time.time()

meta_model = XGBClassifier()
meta_model.fit(meta_features, train_data['binds'])

# End timer
end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds.")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 32 concurrent workers.


Training completed in 7.60 seconds.


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    7.6s finished


In [ ]:
local_model_path = f"./models/{protein_name}_meta_model.pkl"
joblib.dump(meta_model, local_model_path)